# Native facility-location optimization
Preserve three facilities, four customers, exactly two used facilities, all twelve assignment bits, five cost/mixer layers, penalty10 and50optimizationiterations. Keep the full nonlinear empty-facility penalty. Public native exact means and Nelder-Mead replace proprietary shot-based optimization. Actual optimized and zero-angle uniform samples, all source network/trace/histogram/table stages and a fresh exact constrained classical reference remain separate.
The original native nonlinear program has five clean temporary work bits in addition to twelve decisions (seventeen physical bits).


In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.logistics.facility_location.overview.model import (
    CUSTOMERS,
    FACILITIES,
    WEIGHTS,
    assignment,
)
from examples.synthesis.applications.logistics.facility_location.overview.plots import (
    plot_network,
    plot_results,
)
from examples.synthesis.applications.logistics.facility_location.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.logistics.facility_location.overview.workflow import (
    INITIAL_PARAMETERS,
    run_workflow,
)
from flagquantum.drawer import draw

print("Facilities:", FACILITIES, "; customers:", CUSTOMERS)
print("Original delivery-cost matrix:")
for facility_index, facility_name in enumerate(FACILITIES):
    print(facility_name, WEIGHTS[4 * facility_index : 4 * facility_index + 4])
artifacts = Path("artifacts/development/facility-location")
source_graph = plot_network(artifacts / "facility-network.png")
display(Image(filename=str(source_graph)))


In [ ]:
print("Native explicit constrained model replaces Pyomo modeling and display.")
print("Every decision bit indicates one facility/customer assignment.")


In [ ]:
print("Every customer has exactly one supplier; exactly two facilities are used.")
print("A used facility has at least one customer.")
print("Keep the nonlinear squared penalty on the number of empty facilities.")


In [ ]:
decisions = tuple(assignment(word) for word in range(4096))
print("All original decision assignments:", len(decisions))
print("Feasible decision assignments:", sum(value.feasible for value in decisions))
print("No classical solution selected before quantum optimization.")


In [ ]:
initial_program = build_program(INITIAL_PARAMETERS)
print(
    "Original12decisionbits+5cleanworkbits=17physicalbits;fivecost/mixerlayers,penalty10."
)
print("Explicit native exact mean objective; source optimize call omitted quantile.")
print(
    "Reviewed current SDK default is1.0; no historical SDK version or trajectory assertion."
)


In [ ]:
print(initial_program.report["resources"])
print(draw(initial_program.circuit, show_all_qubits=True))


In [ ]:
workflow = run_workflow(maxiter=50, shots=1000, seed=2718)
print("Actual native objective evaluations:", len(workflow.trace))
print("Optimizer iterations:", workflow.optimizer.iterations)


In [ ]:
plot_paths = plot_results(workflow, artifacts)
display(Image(filename=str(plot_paths[1])))


In [ ]:
print("Top five actual source-cost observations:")
for row in workflow.samples[:5]:
    print(row, "source cost=", row.assignment.cost)
assert sum(row.count for row in workflow.samples) == 1000


In [ ]:
print("Actual zero-angle native uniform observations:", len(workflow.uniform))
assert sum(row.count for row in workflow.uniform) == 1000


In [ ]:
display(Image(filename=str(plot_paths[2])))


In [ ]:
best = workflow.best_observed.assignment
print("Best observed full source assignment:", best)


In [ ]:
print("Observed facility/customer table:")
for facility_name, customer_values in zip(FACILITIES, best.rows, strict=True):
    print(facility_name, dict(zip(CUSTOMERS, customer_values, strict=True)))
print("Used facilities:", tuple(FACILITIES[index] for index in best.opened))


In [ ]:
print("Selected sampled assignment remains:", best.word)


In [ ]:
print("Source penalized cost:", best.cost)
print("Raw delivery cost:", best.raw_cost)
print("Penalty:", best.penalty)
print("Feasibility:", best.feasible)
display(Image(filename=str(plot_paths[3])))


In [ ]:
print("Separate exact constrained solve after sampling:")
print(
    "Fresh exhaustive solver replaces external Couenne; no optimizer trajectory claim."
)
print("All tied optimal assignments:", workflow.reference)


In [ ]:
for exact_assignment in workflow.reference:
    print("Exact raw delivery cost:", exact_assignment.raw_cost)
    for facility_name, customer_values in zip(
        FACILITIES, exact_assignment.rows, strict=True
    ):
        print(facility_name, dict(zip(CUSTOMERS, customer_values, strict=True)))
display(Image(filename=str(plot_paths[4])))
report = {
    "decision_bits": 12,
    "workspace_bits": 5,
    "width": 17,
    "layers": 5,
    "maxiter": 50,
    "iterations": workflow.optimizer.iterations,
    "evaluations": len(workflow.trace),
    "quantile": 1,
    "optimized_shots": sum(row.count for row in workflow.samples),
    "uniform_shots": sum(row.count for row in workflow.uniform),
    "best_word": best.word,
    "best_cost": best.cost,
    "best_raw_cost": best.raw_cost,
    "best_penalty": best.penalty,
    "best_feasible": best.feasible,
    "reference_cost": workflow.reference[0].raw_cost,
    "reference_words": [value.word for value in workflow.reference],
    "trace": [value.cost for value in workflow.trace],
}
print("ARTIFACT_JSON=" + json.dumps(report))
